# Kaggriculture — Carrot Throughput Agent

Mantenha este notebook na mesma pasta que `submission_carrot_optimized.py`.

## Janela de colheita

A Carrot normal chega a 3 unidades depois de `WATER` no dia 3. No começo desse dia, ela ainda pode ter 2 unidades; no dia 4, colha o restante antes do decaimento.

In [ ]:
from pathlib import Path
import importlib.util

AGENT_FILE = Path('submission_carrot_optimized.py')
if not AGENT_FILE.exists():
    raise FileNotFoundError(f'Coloque {AGENT_FILE.name} na mesma pasta do notebook.')

spec = importlib.util.spec_from_file_location('carrot_agent', AGENT_FILE)
carrot_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(carrot_agent)
print('Agente carregado:', carrot_agent.agent.__name__)

In [ ]:
# Teste determinístico: não requer kaggle_environments.
plant = {'kind': 'PLANT', 'crop': 'CARROT', 'planted_day': 0, 'watered_today': False, 'yield_units': 2}
day3 = {'day': 3}
assert not carrot_agent._ready_to_harvest(day3, plant)

plant_after_water = {**plant, 'yield_units': 3, 'watered_today': True}
assert carrot_agent._ready_to_harvest(day3, plant_after_water)
assert carrot_agent._ready_to_harvest({'day': 4}, plant)
print('OK: dia 3 / 2 unidades -> esperar WATER')
print('OK: dia 3 / 3 unidades -> HARVEST')
print('OK: dia 4 / unidades restantes -> HARVEST de segurança')

## Simulação local

Instale `kaggle-environments` se necessário. Primeiro rode 240 turnos; depois, altere para 720.

In [ ]:
try:
    from kaggle_environments import make
except ModuleNotFoundError:
    print('Dependência ausente. Instale com: pip install kaggle-environments')
else:
    env = make('kaggriculture', configuration={'episodeSteps': 240}, debug=True)
    env.run([carrot_agent.agent, 'random'])
    final_obs = env.steps[-1][0].observation
    farm = final_obs['farms'][final_obs['player']]
    private = final_obs['private']
    print({'day': final_obs['day'], 'hour': final_obs['hour'], 'money': farm['money'], 'carrots_in_shed': private['shed'].get('CARROT', 0), 'carrot_seeds': private['seeds'].get('CARROT', 0), 'active_hands': len(farm['hands'])})